# Training and order comparison

Use a deterministic sentence-level holdout. Unseen transitions have zero probability without smoothing; report coverage and conditional NLL on supported transitions separately. Do not interpret conditional NLL as full test perplexity.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import markovify
from markovify.nlp import Corpus
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUT = ROOT / "notebooks" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
print(markovify.__file__)

In [ ]:
import random, math, time
from markovify.chain import BEGIN, END
corpus = Corpus.load(OUT / "processed.jsonl")
items = list(corpus)
random.Random(SEED).shuffle(items)
if len(items) < 5:
    raise ValueError("At least five sentences required")
cut = max(1, int(0.8*len(items)))
train, test = Corpus(items[:cut]), Corpus(items[cut:])
train.save(OUT / "train.jsonl")
test.save(OUT / "test.jsonl")
def evaluate(model, sentences):
    k = model.state_size
    total = supported = 0
    nll = 0.0
    for sentence in sentences:
        seq = [BEGIN]*k + sentence.split() + [END]
        for i in range(len(seq)-k):
            state, token = tuple(seq[i:i+k]), seq[i+k]
            counts = model.chain.model.get(state, {})
            total += 1
            if counts.get(token,0) > 0:
                supported += 1
                nll -= math.log(counts[token]/sum(counts.values()))
    return {"coverage":supported/total, "conditional_nll":nll/supported if supported else float("nan"), "full_perplexity":math.exp(nll/total) if supported == total else float("inf")}
rows = []
for k in [1,2,3,4]:
    start = time.perf_counter()
    model = train.to_model(state_size=k, random_state=SEED)
    elapsed = time.perf_counter()-start
    rows.append({"order":k,"states":len(model.chain.model),"fit_seconds":elapsed,**evaluate(model,test)})
results = pd.DataFrame(rows)
display(results)
results.to_csv(OUT / "order_comparison.csv",index=False)

In [ ]:
results.plot(x="order",y="coverage",marker="o",legend=False,ylim=(0,1.05))
plt.ylabel("Held-out transition coverage")
plt.show()
results.plot(x="order",y="states",marker="o",legend=False)
plt.ylabel("Number of states")
plt.show()
# Illustrative fixed order; holdout is diagnostic, not used for tuning.
ORDER = 2
model = train.to_model(state_size=ORDER, random_state=SEED)
model.save(OUT / "model.json.gz", metadata={"order":ORDER,"seed":SEED,"train_sentences":len(train)})
model.compile().save(OUT / "compiled.json.gz")